# Gas Chromatography (GC) for Mixture Analysis

**Objective:** This lesson introduces Gas Chromatography (GC), a powerful technique for separating and quantifying the components of a volatile mixture. We will learn how to interpret a chromatogram, perform peak integration, and use a calibration curve to determine the composition of an unknown sample.

**Learning Goals:**
1.  Understand the basic principles of chromatography: mobile phase, stationary phase, and differential partitioning.
2.  Interpret a **chromatogram**, identifying **retention time** and **peak area**.
3.  Understand that retention time is used for qualitative identification (what is it?) and peak area is used for quantitative analysis (how much is there?).
4.  Use numerical integration (`scipy.integrate.simps`) to calculate peak areas.
5.  Construct and use a calibration curve to find the concentration of a component in a mixture.

## Part 1: The Theory - How GC Works

GC separates components of a mixture by injecting a small amount of the sample into a carrier gas stream (the **mobile phase**). This stream flows through a long, thin tube called a column, which is coated on the inside with a liquid or solid (the **stationary phase**).

Each component in the mixture interacts with the stationary phase differently. Components that interact strongly will move slowly, while components that interact weakly will be swept along by the carrier gas and move quickly. This difference in speed causes the components to separate.

As each separated component leaves the column, it passes through a detector, which generates a signal. The resulting plot of detector signal vs. time is called a **chromatogram**.

*   **Retention Time ($t_R$):** The time it takes for a component to travel through the column. It is characteristic of a specific compound under specific conditions and is used for identification.
*   **Peak Area:** The area under the peak is proportional to the amount (concentration) of that component in the sample.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import simps # Simpson's rule for numerical integration
from scipy import stats

# --- Part 2: Simulate a Chromatogram ---
# In a real scenario, you would load this data from an instrument file.
time = np.linspace(0, 10, 2000) # minutes

# Define parameters for three chemical peaks
# [retention_time, peak_width, amplitude]
peak1_params = [3.5, 0.1, 80]
peak2_params = [5.8, 0.15, 120]
peak3_params = [7.2, 0.2, 50]

def gaussian(x, mu, sigma, amplitude):
    return amplitude * np.exp(-((x - mu) / sigma)**2 / 2)

signal = (gaussian(time, *peak1_params) + 
          gaussian(time, *peak2_params) + 
          gaussian(time, *peak3_params))
signal += np.random.normal(0, 0.5, size=signal.shape) # Add baseline noise

# --- Plot the Chromatogram ---
plt.figure(figsize=(12, 6))
plt.plot(time, signal)
plt.title('Simulated Gas Chromatogram', fontsize=16, weight='bold')
plt.xlabel('Retention Time (minutes)', fontsize=12)
plt.ylabel('Detector Signal (arbitrary units)', fontsize=12)
plt.grid(True)
plt.show()

## Part 3: Peak Integration for Quantification

To find the amount of each component, we need to calculate the area under each peak. We can do this numerically by isolating the data for each peak and using an integration function.

**Note:** Professional chromatography software has sophisticated algorithms for baseline correction and peak deconvolution. We will use a simplified approach by defining manual integration windows.

In [ ]:
# --- Define Integration Windows ---
peak1_window = (time > 3.0) & (time < 4.0)
peak2_window = (time > 5.2) & (time < 6.4)
peak3_window = (time > 6.6) & (time < 7.8)

# --- Integrate using Simpson's Rule ---
# simps(y, x) calculates the area under the curve defined by x and y points.
area1 = simps(signal[peak1_window], time[peak1_window])
area2 = simps(signal[peak2_window], time[peak2_window])
area3 = simps(signal[peak3_window], time[peak3_window])

print("--- Calculated Peak Areas ---")
print(f"Peak 1 (tR={peak1_params[0]} min): Area = {area1:.2f}")
print(f"Peak 2 (tR={peak2_params[0]} min): Area = {area2:.2f}")
print(f"Peak 3 (tR={peak3_params[0]} min): Area = {area3:.2f}")

# --- Visualize the Integration ---
plt.figure(figsize=(12, 6))
plt.plot(time, signal, label='Signal')
plt.fill_between(time[peak1_window], signal[peak1_window], alpha=0.3, label=f'Area 1 = {area1:.1f}')
plt.fill_between(time[peak2_window], signal[peak2_window], alpha=0.3, label=f'Area 2 = {area2:.1f}')
plt.fill_between(time[peak3_window], signal[peak3_window], alpha=0.3, label=f'Area 3 = {area3:.1f}')
plt.title('Chromatogram with Integrated Peaks')
plt.xlabel('Retention Time (minutes)')
plt.ylabel('Detector Signal')
plt.legend()
plt.show()

## Part 4: Calibration and Analysis of an Unknown

Let's say we are interested in the component at $t_R = 5.8$ min (Peak 2). To find its concentration in our sample, we must first run a series of standards with known concentrations and create a calibration curve of **Peak Area vs. Concentration**.

In [ ]:
# --- Calibration Data for Peak 2 ---
standard_concentrations = np.array([10, 20, 50, 80, 100]) # in ppm
standard_areas = np.array([21.5, 43.1, 108.2, 172.5, 215.3])

# This is the area of Peak 2 from our unknown sample
unknown_area = area2

# --- Perform Linear Regression ---
slope, intercept, r_value, _, _ = stats.linregress(standard_concentrations, standard_areas)
r_squared = r_value**2

# --- Plot Calibration Curve ---
plt.figure(figsize=(10, 6))
plt.plot(standard_concentrations, standard_areas, 'bo', label='Standard Data')
fit_line = slope * standard_concentrations + intercept
plt.plot(standard_concentrations, fit_line, 'r-', label=f'Linear Fit (R$^2$={r_squared:.4f})')
plt.title('GC Calibration Curve for Component 2')
plt.xlabel('Concentration (ppm)')
plt.ylabel('Integrated Peak Area')
plt.legend()
plt.grid(True)
plt.show()

# --- Calculate Unknown Concentration ---
# Area = slope * Conc + intercept  => Conc = (Area - intercept) / slope
unknown_concentration = (unknown_area - intercept) / slope

print(f"\nThe calculated concentration of component 2 in the unknown sample is {unknown_concentration:.2f} ppm.")

## Student Challenges

1.  **Overlapping Peaks:** In our simulation, the peaks are perfectly separated (baseline resolved). What happens if two components have very similar retention times? Modify the parameters in Part 2 so that Peak 2 and Peak 3 overlap significantly (e.g., set `peak3_params` retention time to `6.2`). Can our simple window-based integration still work? This illustrates the need for more advanced peak deconvolution algorithms.

2.  **Internal Standard:** A more robust method of quantification uses an 'internal standard' (IS), a known amount of a compound not present in the original sample. You create a calibration curve by plotting (Area_analyte / Area_IS) vs (Conc_analyte / Conc_IS). Why is this method more accurate than the external standard method we used above? (Hint: think about small variations in the injection volume).